# pandas II: Cleaning, Transforming, Grouping, Joining, Reshaping and Time Series

Notebook for **Chapter 5** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch05.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
import pandas as pd, numpy as np, io
raw = """order_id,customer_id,city,order_date,amount,qty
1,C1,pune,2025-01-01,500,2
2,C2,Mumbai ,2025-01-01,1200,1
3,C3,MUMBAI,2025-01-02,,3
4,C1,Pune,2025-01-02,450,1
4,C1,Pune,2025-01-02,450,1
5,C4,Delhi,2025-01-03,99999,2
6,C5,delhi,not_a_date,300,4
7,C2,Mumbai,2025-01-04,"1,150",1
8,C6,Pune,2025-01-05,380,
"""
df = pd.read_csv(io.StringIO(raw), dtype=str)

## 5.1 Cleaning: Missing Values, Duplicates, Types and Text

In [ ]:
print(df.isna().sum())
print(df.duplicated().sum(), df.duplicated(subset="order_id").sum())

order_id       0
customer_id    0
city           0
order_date     0
amount         1
qty            1
dtype: int64
1 1


In [ ]:
df = df.drop_duplicates()
df["city"] = df["city"].str.strip().str.title()
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"].str.replace(",", ""), errors="coerce")
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")
print(df)

  order_id customer_id    city order_date   amount  qty
0        1          C1    Pune 2025-01-01    500.0  2.0
1        2          C2  Mumbai 2025-01-01   1200.0  1.0
2        3          C3  Mumbai 2025-01-02      NaN  3.0
3        4          C1    Pune 2025-01-02    450.0  1.0
5        5          C4   Delhi 2025-01-03  99999.0  2.0
6        6          C5   Delhi        NaT    300.0  4.0
7        7          C2  Mumbai 2025-01-04   1150.0  1.0
8        8          C6    Pune 2025-01-05    380.0  NaN


## 5.2 Transforming Columns

In [ ]:
df["amount_f"] = df["amount"].fillna(500.0)
print(df.assign(tag=np.where(df.amount_f > 1000, "big", "small"))["tag"].tolist())
print(pd.cut(df.amount_f, bins=[0, 400, 1000, np.inf], labels=["low", "mid", "high"]).value_counts())
print(pd.qcut(df.amount_f, q=4, labels=False).tolist())

['small', 'big', 'small', 'small', 'big', 'small', 'big', 'small']
amount_f
mid     3
high    3
low     2
Name: count, dtype: int64
[1, 3, 1, 1, 3, 0, 2, 0]


## 5.3 Grouping and Summarising

In [ ]:
g = df.groupby("city").agg(orders=("order_id", "count"),
                           revenue=("amount", "sum"),
                           avg=("amount", "mean"))
print(g)

In [ ]:
df["city_share"] = df["amount"] / df.groupby("city")["amount"].transform("sum")

In [ ]:
pd.pivot_table(df, index="city", columns=df.order_date.dt.day, values="amount", aggfunc="sum")
pd.crosstab(df.city, df.qty.fillna(0))
df.city.value_counts(normalize=True)

order_date     1.0    2.0      3.0     4.0    5.0
city                                             
Delhi          NaN    NaN  99999.0     NaN    NaN
Mumbai      1200.0    0.0      NaN  1150.0    NaN
Pune         500.0  450.0      NaN     NaN  380.0
qty     0.0  1.0  2.0  3.0  4.0
city                                  
Delhi     0    0    1    0    1
Mumbai    0    2    0    1    0
Pune      1    1    1    0    0
city
Pune      0.375
Mumbai    0.375
Delhi     0.250
Name: proportion, dtype: float64


In [ ]:
d = pd.DataFrame({"day": pd.date_range("2025-01-01", periods=7),
                  "rev": [100, 120, 90, 150, 160, 140, 200]}).set_index("day")
d["ma3"] = d.rev.rolling(3).mean()
d["cum"] = d.rev.cumsum()
d["pct"] = d.rev.pct_change().round(3)
d["prev"] = d.rev.shift(1)
d["rank"] = d.rev.rank(ascending=False)
d["ewm"] = d.rev.ewm(span=3, adjust=False).mean().round(2)
print(d)

            rev         ma3  cum    pct   prev  rank     ewm
day                                                         
2025-01-01  100         NaN  100    NaN    NaN   6.0  100.00
2025-01-02  120         NaN  220  0.200  100.0   5.0  110.00
2025-01-03   90  103.333333  310 -0.250  120.0   7.0  100.00
2025-01-04  150  120.000000  460  0.667   90.0   3.0  125.00
2025-01-05  160  133.333333  620  0.067  150.0   2.0  142.50
2025-01-06  140  150.000000  760 -0.125  160.0   4.0  141.25
2025-01-07  200  166.666667  960  0.429  140.0   1.0  170.62


## 5.4 Combining Tables: merge, concat, join

In [ ]:
a = pd.DataFrame({"cid": ["C1", "C2", "C3"], "name": ["Asha", "Ben", "Chen"]})
b = pd.DataFrame({"cid": ["C2", "C3", "C4", "C3"], "amt": [10, 20, 30, 5]})
print(a.merge(b, on="cid", how="outer", indicator=True))

  cid  name   amt      _merge
0  C1  Asha   NaN   left_only
1  C2   Ben  10.0        both
2  C3  Chen  20.0        both
3  C3  Chen   5.0        both
4  C4   NaN  30.0  right_only


## 5.5 Reshaping: wide, long, nested

In [ ]:
w = pd.DataFrame({"city": ["Pune", "Delhi"], "jan": [10, 20], "feb": [12, 18]})
l = w.melt(id_vars="city", var_name="month", value_name="sales")
print(l)
print(l.pivot(index="city", columns="month", values="sales"))

    city month  sales
0   Pune   jan     10
1  Delhi   jan     20
2   Pune   feb     12
3  Delhi   feb     18
month  feb  jan
city           
Delhi   18   20
Pune    12   10


In [ ]:
e = pd.DataFrame({"id": [1, 2], "tags": [["a", "b"], ["c"]]})
print(e.explode("tags"))
nested = [{"id": 1, "user": {"name": "A", "geo": {"city": "Pune"}}},
          {"id": 2, "user": {"name": "B", "geo": {"city": "Delhi"}}}]
print(pd.json_normalize(nested))

   id tags
0   1    a
0   1    b
1   2    c
   id user.name user.geo.city
0   1         A          Pune
1   2         B         Delhi


## 5.6 Time Series Basics

In [ ]:
ts = pd.DataFrame({"ts": pd.to_datetime(["2025-01-01 10:00", "2025-01-01 10:20",
                                          "2025-01-01 11:05", "2025-01-02 09:00"]),
                   "v": [1, 2, 3, 4]}).set_index("ts")
print(ts.resample("D").agg(["sum", "count"]))
print(ts.index.day_name().tolist())
print(ts.index.tz_localize("UTC").tz_convert("Asia/Kolkata")[0])

             v      
           sum count
ts                  
2025-01-01   6     3
2025-01-02   4     1
['Wednesday', 'Wednesday', 'Wednesday', 'Thursday']
2025-01-01 15:30:00+05:30


## 5.7 Forecasting Basics: Baselines, Smoothing and Honest Backtests

In [ ]:
import numpy as np, pandas as pd, itertools
rng = np.random.default_rng(11)
months = pd.period_range("2021-01", periods=48, freq="M")
season = np.array([-8, -10, -4, 0, 3, 5, 4, 6, 2, 10, 18, 22], dtype=float)
y = pd.Series((100 + 1.5 * np.arange(48) + np.tile(season, 4) + rng.normal(0, 3, 48)).round(1), index=months)
train, test = y.iloc[:36], y.iloc[36:]

def hw(x, h, m, a, b, g):
    """Additive Holt-Winters. Returns (one-step squared error on x, forecast of h steps)."""
    level = x[:m].mean(); trend = (x[m:2*m].mean() - x[:m].mean()) / m
    seas = list(x[:m] - level); sse = 0.0
    for t in range(m, len(x)):
        pred = level + trend + seas[t - m]; sse += (x[t] - pred) ** 2
        prev = level
        level = a * (x[t] - seas[t - m]) + (1 - a) * (level + trend)
        trend = b * (level - prev) + (1 - b) * trend
        seas.append(g * (x[t] - level) + (1 - g) * seas[t - m])
    return sse, np.array([level + (k + 1) * trend + seas[len(x) - m + (k % m)] for k in range(h)])

grid = list(itertools.product((0.1, 0.3, 0.5), (0.05, 0.1, 0.2), (0.1, 0.3, 0.5)))
best = min(grid, key=lambda p: hw(train.values, 12, 12, *p)[0])
print("chosen alpha, beta, gamma:", best)

forecasts = {
    "naive": np.repeat(train.iloc[-1], 12),
    "seasonal naive": train.iloc[-12:].values,
    "3-month average": np.repeat(train.iloc[-3:].mean(), 12),
    "Holt-Winters": hw(train.values, 12, 12, *best)[1],
}
rows = []
for name, f in forecasts.items():
    e = np.abs(test.values - f)
    rows.append({"method": name, "MAE": e.mean(), "MAPE_%": (e / test.values).mean() * 100,
                 "WAPE_%": e.sum() / test.values.sum() * 100})
print(pd.DataFrame(rows).round(2).to_string(index=False))

chosen alpha, beta, gamma: (0.5, 0.05, 0.5)
         method   MAE  MAPE_%  WAPE_%
          naive 12.71    7.94    7.67
 seasonal naive 18.73   11.49   11.31
3-month average 10.46    6.41    6.31
   Holt-Winters  7.12    4.41    4.30


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(11)
months = pd.period_range("2021-01", periods=48, freq="M")
season = np.array([-8, -10, -4, 0, 3, 5, 4, 6, 2, 10, 18, 22], dtype=float)
y = pd.Series((100 + 1.5 * np.arange(48) + np.tile(season, 4) + rng.normal(0, 3, 48)).round(1), index=months)
train = y.iloc[:36]
print(train.iloc[-3:].to_string())
print("3-month average:", round(train.iloc[-3:].mean(), 2))
print("pandas rolling :", round(train.rolling(3).mean().iloc[-1], 2))

2023-10    160.7
2023-11    171.2
2023-12    173.6
Freq: M
3-month average: 168.5
pandas rolling : 168.5


## 5.8 Pandas Recipes for Experts: Group-wise Windows, Chaining and Safe Merges

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "city": ["Pune", "Pune", "Pune", "Delhi", "Delhi", "Mumbai", "Mumbai", "Mumbai"],
    "amount": [1200, 800, 2000, 1500, 1100, 700, 1300, 600],
    "status": ["delivered"] * 7 + ["returned"]})
g = orders.groupby("city")["amount"]
out = orders.assign(
    city_total=g.transform("sum"),
    share_pct=lambda d: (d["amount"] / d["city_total"] * 100).round(1),
    rank_in_city=g.rank(ascending=False, method="dense").astype(int),
    vs_city_mean=lambda d: (d["amount"] - g.transform("mean")).round(1))
print(out.to_string(index=False))

 order_id   city  amount    status  city_total  share_pct  rank_in_city  vs_city_mean
        1   Pune    1200 delivered        4000       30.0             2        -133.3
        2   Pune     800 delivered        4000       20.0             3        -533.3
        3   Pune    2000 delivered        4000       50.0             1         666.7
        4  Delhi    1500 delivered        2600       57.7             1         200.0
        5  Delhi    1100 delivered        2600       42.3             2        -200.0
        6 Mumbai     700 delivered        2600       26.9             2        -166.7
        7 Mumbai    1300 delivered        2600       50.0             1         433.3
        8 Mumbai     600  returned        2600       23.1             3        -266.7


In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "city": ["Pune", "Pune", "Pune", "Delhi", "Delhi", "Mumbai", "Mumbai", "Mumbai"],
    "amount": [1200, 800, 2000, 1500, 1100, 700, 1300, 600],
    "status": ["delivered"] * 7 + ["returned"]})

def add_gst(d, rate):
    return d.assign(amount_with_gst=d["amount"] * (1 + rate))

result = (orders
          .query("status == 'delivered'")
          .pipe(add_gst, rate=0.18)
          .groupby("city", as_index=False)
          .agg(orders=("order_id", "count"), revenue=("amount_with_gst", "sum"))
          .sort_values("revenue", ascending=False))
print(result.to_string(index=False))

  city  orders  revenue
  Pune       3   4720.0
 Delhi       2   3068.0
Mumbai       2   2360.0


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import pandas as pd
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "city": ["Pune", "Pune", "Pune", "Delhi", "Delhi", "Mumbai", "Mumbai", "Mumbai"],
    "amount": [1200, 800, 2000, 1500, 1100, 700, 1300, 600]})
zones = pd.DataFrame({"city": ["Pune", "Pune", "Delhi"], "zone": ["West", "West-2", "North"]})

inner = orders.merge(zones, on="city")
print("inner rows:", len(inner), "| revenue:", inner["amount"].sum(), "| true revenue:", orders["amount"].sum())
left = orders.merge(zones, on="city", how="left", indicator=True)
print("left rows:", len(left), "| unmatched:", int((left["_merge"] == "left_only").sum()))
try:
    orders.merge(zones, on="city", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__ + ":", str(e).splitlines()[0])

inner rows: 8 | revenue: 10600 | true revenue: 9200
left rows: 11 | unmatched: 3
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import pandas as pd
rates = pd.DataFrame({
    "effective": pd.to_datetime(["2025-01-01 00:00", "2025-02-01 00:00", "2025-03-01 00:00"]).tz_localize("UTC"),
    "rate": [0.05, 0.08, 0.10]})
ev = pd.DataFrame({
    "ts": pd.to_datetime(["2025-01-15 10:00", "2025-02-01 02:00", "2025-02-28 23:30"]).tz_localize("Asia/Kolkata"),
    "amount": [100, 200, 300]})
try:
    pd.merge_asof(ev, rates, left_on="ts", right_on="effective")
except Exception as e:
    print(type(e).__name__ + ":", str(e)[:75])
ev["ts_utc"] = ev["ts"].dt.tz_convert("UTC")
print(pd.merge_asof(ev, rates, left_on="ts_utc", right_on="effective")[["ts", "ts_utc", "rate"]].to_string(index=False))

MergeError: incompatible merge keys [0] datetime64[us, Asia/Kolkata] and datetime64[us,
                       ts                    ts_utc  rate
2025-01-15 10:00:00+05:30 2025-01-15 04:30:00+00:00  0.05
2025-02-01 02:00:00+05:30 2025-01-31 20:30:00+00:00  0.05
2025-02-28 23:30:00+05:30 2025-02-28 18:00:00+00:00  0.08


## Chapter recap

## Practice set

Problem 1 `pd.Series([1, np.nan, 3]).interpolate()` gives what?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch05.html